# 02e — Frozen Foundation-Model Head (anggota ensemble ke-5)

**Ide inti**: backbone besar (DINOv2-Large secara default, atau CLIP ViT-B/32) DIBEKUKAN total —
tidak ada backward pass lewat backbone sama sekali. Fitur untuk seluruh 26k gambar train diekstrak
SEKALI (satu forward pass, no_grad), lalu di-cache ke Drive sebagai `.npy`. Yang benar-benar dilatih
per fold hanyalah kepala klasifikasi kecil (MLP 2 layer) di atas fitur yang sudah tetap itu —
karena itu seluruh loop 5 fold selesai dalam hitungan menit, bukan jam seperti MaxViT/SwinV2.

**Kenapa ini relevan untuk masalah yang sudah didiagnosis**: OOF confusion matrix 14-model
menunjukkan 64 dari ~79 kesalahan ensemble ada di batas Organic<->Recyclable (0<->2), dan
`00_Diagnostik_Kemiripan_TrainTest.ipynb` menemukan pola konkret (bunga kertas/tisu, tirai kain,
pipet) yang mirip SECARA VISUAL dengan satu kelas tapi SECARA MATERIAL kelas lain — sesuatu yang
sulit dipisahkan oleh CNN/ViT yang di-fine-tune penuh dengan tujuan klasifikasi 3-kelas kita
sendiri (representasinya terbentuk mengikuti shortcut visual yang cukup untuk 26k data training).
Foundation model besar dilatih dengan objektif yang berbeda skalanya (DINOv2: self-supervised
skala sangat besar, sensitif struktur semantik/part-level; CLIP: kontrastif image-text, caption
manusia sering menyebut material eksplisit) — errornya diharapkan TIDAK berkorelasi dengan error
4 arsitektur lama, sehingga anggota ke-5 ini punya nilai ensemble walau akurasi standalone-nya
mungkin tidak setinggi ConvNeXtV2/SwinV2.

**Batas kepatuhan PRD yang WAJIB dijaga (§2 rule 3, §11)**: notebook ini HANYA membuka gambar dari
`Preprocessing_Output/processed/` (train). Tidak ada satu baris kode pun di sini yang menyentuh
folder `test/` — itu murni hak `04_Inference_and_Submission.ipynb`. Ekstraksi fitur test + inferensi
kepala untuk anggota ke-5 ini SENGAJA ditunda ke pembaruan NB04 di masa depan (belum dibuat di sini)
supaya batas "test hanya boleh disentuh NB04" tidak pernah dilanggar oleh notebook mana pun.

**Catatan label**: `true_label` yang ditulis ke OOF di sini memakai label MENTAH
`fold_assignment.csv` (belum dikoreksi `all_label_corrections.csv`) — supaya konsisten dengan OOF
4 arsitektur lain yang saat ini masih berbasis checkpoint Stage 1-3 (label lama). Begitu seluruh
tim mengadopsi checkpoint Stage 3.5 (label terkoreksi) untuk keempat arsitektur itu, notebook ini
HARUS dijalankan ulang dengan `RELABEL_CSV` diterapkan juga, supaya `true_label` tetap seragam
lintas seluruh anggota ensemble (lihat catatan yang sama di `02x_Finetune_Stage35.ipynb`).


In [ ]:
# transformers hanya perlu di-install kalau FEATURE_SOURCE='clip_vitb32'; timm (untuk DINOv2) sudah
# lazim terpasang di image Kaggle/Colab tapi kita pin versi yang sama dengan 02x/02a-d untuk
# konsistensi tag model (deviasi dari PRD §12 dicatat di notebook lain, alasan sama: image
# GPU sudah bawa torch/torchvision yang cocok dengan driver CUDA-nya).
!pip install -q timm==1.0.11 transformers==4.46.3


In [ ]:
import hashlib
import json
from pathlib import Path

import numpy as np
import pandas as pd
import timm
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image
from sklearn.metrics import f1_score
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
NUM_CLASSES = 3
N_FOLDS = 5
print("device:", DEVICE, "| torch:", torch.__version__, "| timm:", timm.__version__)


## Config

`FEATURE_SOURCE` yang menentukan backbone beku: `'dinov2_large'` (default, direkomendasikan
duluan -- linear-probe DINOv2 secara umum unggul dibanding CLIP linear-probe untuk tugas
fine-grained di literatur) atau `'clip_vitb32'` (alternatif/pembanding, lebih ringan tapi
dilatih dengan objektif kontrastif teks-gambar yang punya alasan berbeda untuk cocok dengan
masalah kita -- lihat markdown di atas). Jalankan salah satu dulu; kalau waktu memungkinkan,
jalankan keduanya sebagai DUA anggota ensemble terpisah (`foundation_dinov2_large` dan
`foundation_clip_vitb32`) -- NB03 akan menimbang otomatis, bahkan bisa memberi bobot ~0 ke
salah satu kalau tidak membantu (guardrail PRD §10.2 yang sudah ada).


In [ ]:
FEATURE_SOURCE = "dinov2_large"  # atau "clip_vitb32"

FEATURE_SPECS = {
    "dinov2_large": {
        "kind": "timm",
        "timm_tag": "vit_large_patch14_dinov2.lvd142m",
        "feat_dim": 1024,
    },
    "clip_vitb32": {
        "kind": "clip",
        "hf_tag": "openai/clip-vit-base-patch32",
        "feat_dim": 512,
    },
}
SPEC = FEATURE_SPECS[FEATURE_SOURCE]
ARCH_NAME = f"foundation_{FEATURE_SOURCE}"

MASTER_DATASET_VERSION = 1  # harus sama dengan yang dipin di CONFIG 02a-02d (PRD §6.3)
# Notebook ini tidak berbagi kode dengan skeleton 02x (arsitektur trainingnya beda total: backbone
# beku + head kecil, bukan fine-tune penuh). String di bawah dipin SAMA PERSIS dengan SKELETON_VERSION
# di skeleton_body.txt murni sebagai stempel kompatibilitas "pakai fold_assignment.csv + master
# dataset versi yang sama" untuk gate integritas NB03 -- BUKAN klaim berbagi logika training.
SKELETON_VERSION_PIN = "skeleton-v1.3.0"

print(f"FEATURE_SOURCE={FEATURE_SOURCE} | ARCH_NAME={ARCH_NAME} | feat_dim={SPEC['feat_dim']}")


## Mount Drive & path (pola sama dengan `00_Diagnostik_Kemiripan_TrainTest.ipynb` / `02x`)


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_ROOT = Path('/content/drive/MyDrive/Big Data Challenge - Satria Data 2026')
PROCESSED_ROOT = DRIVE_ROOT / 'Preprocessing_Output' / 'processed'
MANIFEST_DIR   = DRIVE_ROOT / 'Preprocessing_Output' / 'manifests'
OOF_ROOT       = DRIVE_ROOT / 'Preprocessing_Output' / 'oof'  # NB03 baca langsung dari sini, tidak perlu upload manual
FOUNDATION_ROOT = DRIVE_ROOT / 'Training_Output_Foundation' / ARCH_NAME
FOUNDATION_ROOT.mkdir(parents=True, exist_ok=True)
OOF_ROOT.mkdir(parents=True, exist_ok=True)

FOLD_ASSIGNMENT_PATH = MANIFEST_DIR / 'fold_assignment.csv'
FEATURES_NPY = FOUNDATION_ROOT / 'train_features.npy'
FEATURE_IDS_NPY = FOUNDATION_ROOT / 'train_feature_ids.npy'

for p in [PROCESSED_ROOT, FOLD_ASSIGNMENT_PATH]:
    assert Path(p).exists(), f"path wajib tidak ditemukan: {p}"

fold_assignment = pd.read_csv(FOLD_ASSIGNMENT_PATH)
print(f"fold_assignment.csv: {len(fold_assignment)} baris, kolom {list(fold_assignment.columns)}")


def file_md5(path, chunk_size=1 << 20):
    h = hashlib.md5()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(chunk_size), b""):
            h.update(chunk)
    return h.hexdigest()


def hash_config(cfg: dict) -> str:
    return hashlib.sha256(json.dumps(cfg, sort_keys=True, default=str).encode()).hexdigest()[:16]


PROVENANCE = {
    "fold_assignment_md5": file_md5(FOLD_ASSIGNMENT_PATH),
    "master_dataset_version": MASTER_DATASET_VERSION,
    "skeleton_version": SKELETON_VERSION_PIN,
    "config_hash": hash_config({"ARCH_NAME": ARCH_NAME, "FEATURE_SOURCE": FEATURE_SOURCE, **SPEC}),
}
print("provenance:", PROVENANCE)


## Bangun backbone beku + fungsi ekstraksi fitur

`torch.no_grad()` di seluruh forward pass -- tidak ada gradien yang pernah mengalir ke backbone.
`model.eval()` dipanggil sekali dan tidak pernah diubah balik ke `.train()` untuk backbone ini.


In [ ]:
class RawImageIdDataset(Dataset):
    def __init__(self, image_ids, transform):
        self.image_ids = list(image_ids)
        self.transform = transform

    def __len__(self):
        return len(self.image_ids)

    def __getitem__(self, idx):
        image_id = int(self.image_ids[idx])
        path = f"{PROCESSED_ROOT}/{image_id}.jpg"
        with Image.open(path) as im:
            tensor = self.transform(im.convert("RGB"))
        return tensor, image_id


if SPEC["kind"] == "timm":
    backbone = timm.create_model(SPEC["timm_tag"], pretrained=True, num_classes=0).to(DEVICE).eval()
    for p in backbone.parameters():
        p.requires_grad_(False)
    data_cfg = timm.data.resolve_data_config({}, model=backbone)
    extract_transform = timm.data.create_transform(**data_cfg, is_training=False)

    @torch.no_grad()
    def extract_batch(images_tensor):
        return backbone(images_tensor.to(DEVICE)).cpu().numpy()

elif SPEC["kind"] == "clip":
    from transformers import CLIPModel, CLIPProcessor
    clip_model = CLIPModel.from_pretrained(SPEC["hf_tag"]).to(DEVICE).eval()
    clip_processor = CLIPProcessor.from_pretrained(SPEC["hf_tag"])
    for p in clip_model.parameters():
        p.requires_grad_(False)
    _clip_img_mean = clip_processor.image_processor.image_mean
    _clip_img_std = clip_processor.image_processor.image_std
    _clip_size = clip_processor.image_processor.crop_size["height"]

    import torchvision.transforms as T
    extract_transform = T.Compose([
        T.Resize(_clip_size, interpolation=T.InterpolationMode.BICUBIC),
        T.CenterCrop(_clip_size),
        T.ToTensor(),
        T.Normalize(mean=_clip_img_mean, std=_clip_img_std),
    ])

    @torch.no_grad()
    def extract_batch(images_tensor):
        vision_out = clip_model.vision_model(pixel_values=images_tensor.to(DEVICE))
        feat = clip_model.visual_projection(vision_out.pooler_output)
        return F.normalize(feat, dim=1).cpu().numpy()

else:
    raise ValueError(f"FEATURE_SOURCE tidak dikenal: {FEATURE_SOURCE}")

print(f"Backbone {SPEC.get('timm_tag') or SPEC.get('hf_tag')} siap (frozen, eval mode).")


## Ekstraksi fitur train (SEKALI, di-cache ke Drive)

Kalau `train_features.npy` sudah ada dan cocok jumlah barisnya dengan `fold_assignment.csv`,
langsung dipakai ulang -- tidak pernah ekstraksi dua kali untuk konfigurasi yang sama.


In [ ]:
if FEATURES_NPY.exists() and FEATURE_IDS_NPY.exists():
    train_features = np.load(FEATURES_NPY)
    train_feature_ids = np.load(FEATURE_IDS_NPY)
    if len(train_feature_ids) == len(fold_assignment):
        print(f"Cache fitur ditemukan: {train_features.shape} -- ekstraksi dilewati.")
    else:
        print("Cache fitur tidak cocok jumlah barisnya dengan fold_assignment.csv saat ini -- ekstraksi ulang.")
        train_features = None
else:
    train_features = None

if train_features is None:
    all_ids = fold_assignment["image_id"].values
    ds = RawImageIdDataset(all_ids, extract_transform)
    loader = DataLoader(ds, batch_size=64, shuffle=False, num_workers=2, pin_memory=(DEVICE.type == "cuda"))

    train_features = np.zeros((len(all_ids), SPEC["feat_dim"]), dtype=np.float32)
    train_feature_ids = np.zeros(len(all_ids), dtype=np.int64)
    ptr = 0
    for images, ids in tqdm(loader, desc=f"Ekstraksi fitur {FEATURE_SOURCE}"):
        feats = extract_batch(images)
        n = len(ids)
        train_features[ptr:ptr + n] = feats
        train_feature_ids[ptr:ptr + n] = ids.numpy()
        ptr += n

    np.save(FEATURES_NPY, train_features)
    np.save(FEATURE_IDS_NPY, train_feature_ids)
    print(f"Ekstraksi selesai: {train_features.shape} disimpan ke {FEATURES_NPY}")

feature_lookup = dict(zip(train_feature_ids.tolist(), range(len(train_feature_ids))))
label_lookup = dict(zip(fold_assignment["image_id"], fold_assignment["label"]))
fold_lookup = dict(zip(fold_assignment["image_id"], fold_assignment["fold"]))


## Head kecil (MLP 2 layer) di atas fitur beku

Ini satu-satunya bagian yang benar-benar "dilatih" per fold. Input sudah berupa vektor tetap
(bukan gambar) jadi tidak ada I/O gambar maupun forward-pass backbone di loop ini sama sekali --
karena itu satu fold selesai dalam hitungan detik-menit walau di CPU sekalipun.


In [ ]:
class FeatureHead(nn.Module):
    def __init__(self, in_dim, hidden=256, num_classes=3, p_drop=0.3):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden),
            nn.GELU(),
            nn.Dropout(p_drop),
            nn.Linear(hidden, num_classes),
        )

    def forward(self, x):
        return self.net(x)


def make_fold_tensors(fold_k):
    train_ids = fold_assignment.loc[fold_assignment["fold"] != fold_k, "image_id"].values
    val_ids = fold_assignment.loc[fold_assignment["fold"] == fold_k, "image_id"].values

    def _gather(ids):
        idx = [feature_lookup[i] for i in ids]
        X = torch.from_numpy(train_features[idx]).float()
        y = torch.tensor([label_lookup[i] for i in ids], dtype=torch.long)
        return X, y, ids

    return _gather(train_ids), _gather(val_ids)


@torch.no_grad()
def eval_head(head, X, y):
    head.eval()
    logits = head(X.to(DEVICE))
    probs = F.softmax(logits, dim=1).cpu().numpy()
    preds = probs.argmax(1)
    macro_f1 = f1_score(y.numpy(), preds, average="macro")
    return macro_f1, probs


HEAD_EPOCHS = 300
HEAD_LR = 1e-3
HEAD_HIDDEN = 256
HEAD_DROPOUT = 0.3
PATIENCE = 30


## Loop 5 fold -- latih head, evaluasi, ekspor OOF sesuai skema kanonis PRD §5.3


In [ ]:
results_per_fold = []

for fold_k in range(N_FOLDS):
    oof_path = OOF_ROOT / f"oof_{ARCH_NAME}_fold{fold_k}.csv"
    if oof_path.exists():
        print(f"[fold {fold_k}] OOF sudah ada -- dilewati")
        continue

    (X_train, y_train, _), (X_val, y_val, val_ids) = make_fold_tensors(fold_k)
    X_train, y_train = X_train.to(DEVICE), y_train.to(DEVICE)

    torch.manual_seed(SEED + fold_k)
    head = FeatureHead(SPEC["feat_dim"], HEAD_HIDDEN, NUM_CLASSES, HEAD_DROPOUT).to(DEVICE)
    optimizer = torch.optim.AdamW(head.parameters(), lr=HEAD_LR, weight_decay=1e-2)
    criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

    best_f1 = -1.0
    best_state = None
    epochs_since_improve = 0

    for epoch in range(1, HEAD_EPOCHS + 1):
        head.train()
        optimizer.zero_grad(set_to_none=True)
        loss = criterion(head(X_train), y_train)
        loss.backward()
        optimizer.step()

        val_f1, _ = eval_head(head, X_val, y_val)
        if val_f1 > best_f1:
            best_f1 = val_f1
            best_state = {k: v.clone() for k, v in head.state_dict().items()}
            epochs_since_improve = 0
        else:
            epochs_since_improve += 1
            if epochs_since_improve >= PATIENCE:
                break

    head.load_state_dict(best_state)
    final_f1, final_probs = eval_head(head, X_val, y_val)
    print(f"[fold {fold_k}] epoch berhenti={epoch} | val macro-F1 terbaik={final_f1:.5f}")

    oof_rows = []
    for i, image_id in enumerate(val_ids):
        p = final_probs[i]
        oof_rows.append({
            "image_id": int(image_id), "prob_0": float(p[0]), "prob_1": float(p[1]), "prob_2": float(p[2]),
            "true_label": int(label_lookup[image_id]), "fold": fold_k,
        })
    oof_df = pd.DataFrame(oof_rows).sort_values("image_id").reset_index(drop=True)
    oof_df.to_csv(oof_path, index=False)

    sidecar = {**PROVENANCE, "arch": ARCH_NAME, "fold": fold_k, "val_macro_f1": float(final_f1),
               "head_hidden": HEAD_HIDDEN, "head_dropout": HEAD_DROPOUT}
    with open(OOF_ROOT / f"oof_{ARCH_NAME}_fold{fold_k}.json", "w") as f:
        json.dump(sidecar, f, indent=2)

    fold_ckpt_dir = FOUNDATION_ROOT / f"fold{fold_k}"
    fold_ckpt_dir.mkdir(parents=True, exist_ok=True)
    torch.save({
        "head_state": best_state, "feat_dim": SPEC["feat_dim"], "hidden": HEAD_HIDDEN,
        "feature_source": FEATURE_SOURCE, "val_macro_f1": float(final_f1), **PROVENANCE,
    }, fold_ckpt_dir / "head.ckpt")

    results_per_fold.append(final_f1)
    print(f"menulis {oof_path}")

if results_per_fold:
    print(f"\n{ARCH_NAME}: {len(results_per_fold)} fold baru selesai, rata-rata val macro-F1={np.mean(results_per_fold):.5f}")
else:
    print(f"\n{ARCH_NAME}: semua 5 fold OOF sudah ada sebelumnya, tidak ada yang dilatih ulang.")


## Langkah lanjutan (belum dilakukan di sini, catatan untuk sesi berikutnya)

1. Cek `oof_report.md`/bobot Nelder-Mead di NB03 setelah menambahkan `"{ARCH_NAME}"` ke daftar
   `ARCHS` -- kalau bobotnya mendekati nol, itu sinyal sah bahwa anggota ini tidak menambah nilai
   untuk data ini (guardrail PRD §10.2 yang sudah ada, bukan kegagalan proses).
2. Kalau bobotnya positif dan cukup besar: perlu pembaruan NB04 (satu-satunya notebook yang boleh
   membuka `test/`) untuk (a) ekstraksi fitur test dengan backbone beku yang sama, (b) memuat
   `head.ckpt` per fold dari `Training_Output_Foundation/{ARCH_NAME}/foldK/`, (c) rata-rata
   probabilitas 5 fold seperti anggota lain -- ini BELUM diimplementasikan di notebook manapun,
   sengaja ditunda supaya PRD §11 (test hanya disentuh NB04) tidak pernah dilanggar di sini.
3. Opsional: ulangi notebook ini dengan `FEATURE_SOURCE = "clip_vitb32"` sebagai anggota keenam
   yang terpisah (`foundation_clip_vitb32`), lalu biarkan NB03 menimbang keduanya sekaligus.
